# Attention Map Defense — Unilingual Sole Two-Box (EN / ZH)

Tests whether ViT **self-attention weights** can replace GradCAM as the saliency signal
for the CAM-intersection defense against **sole-language dual-box** typographic attacks
(both boxes English, or both boxes Chinese) — the unilingual counterpart to the
multilingual notebook one level up.

## Inference cost comparison

Each "pass" below is one forward **or** backward operation (both counted as one pass unit).
GradCAM requires a backward pass per model; attention does not.

| Method | Pass 1 | Pass 2 | Pass 3 | Pass 4 | Pass 5 | Pass 6 | Total |
|---|---|---|---|---|---|---|---|
| GradCAM cam_2mod | EN **fwd** (classify + GradCAM hook) | EN **back** (GradCAM grad) | ZH **fwd** (classify + GradCAM hook) | ZH **back** (GradCAM grad) | EN re-classify | ZH re-classify | **6** |
| **Attn cam_2mod** | EN **fwd** (classify + attn hook) | ZH **fwd** (classify + attn hook) | EN re-classify | ZH re-classify | — | — | **4** |

Classification is **fused** with saliency extraction in both cases — there is no separate
pre-classification step. The difference is that GradCAM needs an additional backward pass
per model (to compute gradients), while attention weights are available from the forward pass
at no extra cost.

Minimum cost breakdown:
```
GradCAM cam_2mod : 2 × (fwd + back)  +  2 × fwd  =  4 + 2  = 6
Attn    cam_2mod : 2 × fwd            +  2 × fwd  =  2 + 2  = 4
```

## Two attention variants
- **last-layer**: CLS→patch attention row from the final transformer block, heads averaged.
- **rollout**: Abnar & Zuidema (2020) — multiply `(0.5·A + 0.5·I)` across all layers.

## Attack
Sole two-box unilingual: same target-class word in both non-overlapping boxes
(`attack_lang='en'` or `'zh'`). Geometry matches the multilingual notebook /
`en_zh_multiple_placement` dual-box setup (`NUM_BOXES=2`, `FONT_SIZE=24`).

Models: EN CLIP `open_clip` ViT-B/32 (8 heads, 7×7 = 49 patches),
ZH CLIP `ChineseCLIP` ViT-B/16 (12 heads, 14×14 = 196 patches).

Run on the **100-image tune subset** first (per attack lang); full 1000-image eval if promising.


In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'open_clip_torch', 'transformers', 'datasets', 'matplotlib', 'Pillow'],
               check=False)

CompletedProcess(args=['d:\\ian\\2026summer\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'open_clip_torch', 'transformers', 'datasets', 'matplotlib', 'Pillow'], returncode=0)

In [2]:
import os, platform, random, json, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont
from datasets import load_dataset
from transformers import ChineseCLIPModel, ChineseCLIPProcessor

os.makedirs('results', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8

CLASSES = {
    'en': ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck'],
    'zh': ['飞机','汽车','鸟','猫','鹿','狗','青蛙','马','船','卡车'],
}
TMPL = {'en': 'a photo of a {}.', 'zh': '一张{}的照片。'}
LANGS = ['en', 'zh']

Device: cuda


## 1. Models

In [3]:
def _clip_feat(out):
    if torch.is_tensor(out): return out
    if getattr(out, 'pooler_output', None) is not None: return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    def __init__(self):
        self.m, _, self.pp = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer('ViT-B-32')
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class ZhCLIP:
    lang = 'zh'
    def __init__(self):
        self.m = ChineseCLIPModel.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16',
                                                   attn_implementation='eager').to(DEVICE).eval()
        self.p = ChineseCLIPProcessor.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['zh'].format(w) for w in words], padding=True, return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'], attention_mask=t['attention_mask'],
                                        token_type_ids=t.get('token_type_ids'))
        return F.normalize(_clip_feat(out), dim=-1)

def classify_batch(model, imgs, words, batch_size=128):
    """Batched classification — used for fast baseline metrics only."""
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i+batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

models = {}
for lang, cls in [('en', EnCLIP), ('zh', ZhCLIP)]:
    t0 = time.time()
    print(f'Loading {lang}...', end=' ', flush=True)
    models[lang] = cls()
    print(f'{time.time()-t0:.1f}s')

TEXT_EMB = {lang: models[lang].embed_texts(CLASSES[lang]).detach() for lang in LANGS}
print('Models ready.')

Loading en... 

d:\ian\2026summer\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


1.4s
Loading zh... 

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

1.4s
Models ready.


## 2. Dataset + sole two-box unilingual attack (EN / ZH)


In [4]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

_sample_path = '../../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json'
with open(_sample_path, encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
rows = hf.select(idx)
true = np.array(rows[label_key])
assert len(idx) == 1000 and np.array_equal(true, np.array(_saved['true']))

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])])
                   for k in range(len(idx))])

clean_224 = [im.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
             for im in rows[image_key]]

all_idx  = np.arange(len(clean_224))
tune_idx = np.concatenate([np.where(true == c)[0][:10] for c in range(10)])
print(f'Loaded {len(clean_224)} images; tune subset = {len(tune_idx)}')


Loaded 1000 images; tune subset = 100


In [5]:
# ─── font helpers ────────────────────────────────────────────────────────────
_FONT_CACHE = {}
def _font_paths():
    if platform.system() == 'Windows':
        wf = os.path.join(os.environ.get('WINDIR', r'C:\Windows'), 'Fonts')
        return (os.path.join(wf, 'msyh.ttc'), os.path.join(wf, 'arial.ttf'))
    return None, None
_CJK_FONT, _LAT_FONT = _font_paths()

def _font_for(word):
    return _CJK_FONT if any(ord(c) > 127 for c in word) else _LAT_FONT

def _get_font(fp, size=FONT_SIZE):
    key = (fp or '__default__', size)
    if key not in _FONT_CACHE:
        try:    _FONT_CACHE[key] = ImageFont.truetype(fp, size) if fp else ImageFont.load_default()
        except: _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]

def _rects_overlap(a, b):
    return not (a[2]<=b[0] or b[2]<=a[0] or a[3]<=b[1] or b[3]<=a[1])

def _random_nonoverlapping_rect(rng_, bw, bh, placed):
    x_hi = max(0, DISPLAY_SIZE-bw); y_hi = max(0, DISPLAY_SIZE-bh)
    rx = ry = 0
    for _ in range(64):
        rx = rng_.randint(0, x_hi) if x_hi > 0 else 0
        ry = rng_.randint(0, y_hi) if y_hi > 0 else 0
        rect = (rx, ry, rx+bw, ry+bh)
        if all(not _rects_overlap(rect, p) for p in placed): return rect
    return (rx, ry, rx+bw, ry+bh)

def draw_word(img, word, img_idx, already_224=False):
    """Place the same word in NUM_BOXES non-overlapping positions."""
    if not already_224:
        img = img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
    else:
        img = img.copy()
    draw = ImageDraw.Draw(img)
    font = _get_font(_font_for(word))
    bb   = draw.textbbox((0, 0), word, font=font)
    bw   = (bb[2]-bb[0]) + 2*PAD
    bh   = (bb[3]-bb[1]) + PAD + 12
    placed = []
    for box_i in range(NUM_BOXES):
        rng_ = random.Random(int(img_idx)*NUM_BOXES + box_i)
        rect = _random_nonoverlapping_rect(rng_, bw, bh, placed)
        placed.append(rect)
        rx, ry, rx2, ry2 = rect
        draw.rectangle([rx, ry, rx2, ry2], fill='white')
        draw.text((rx+PAD-bb[0], ry+PAD-bb[1]), word, fill='black', font=font)
    return img

def build_attacked_images(base_imgs, img_indices, attack_lang):
    """Sole two-box unilingual attack: same word in both boxes."""
    return [
        draw_word(base_imgs[int(k)], CLASSES[attack_lang][target[int(k)]],
                  int(k), already_224=True)
        for k in img_indices
    ]

attacked_by_lang = {}
preds_attacked   = {}
baseline_acc     = {}
baseline_asr     = {}

clean_acc = {ml: float((classify_batch(models[ml], clean_224, CLASSES[ml]) == true).mean())
             for ml in LANGS}
print('Clean acc:   ', {k: f'{100*v:.1f}%' for k, v in clean_acc.items()})

for attack_lang in LANGS:
    t0 = time.time()
    print(f'Building {attack_lang} sole two-box attack...', end=' ', flush=True)
    attacked_by_lang[attack_lang] = build_attacked_images(clean_224, all_idx, attack_lang)
    preds_attacked[attack_lang] = {
        ml: classify_batch(models[ml], attacked_by_lang[attack_lang], CLASSES[ml])
        for ml in LANGS
    }
    baseline_acc[attack_lang] = {
        ml: float((preds_attacked[attack_lang][ml] == true).mean()) for ml in LANGS
    }
    baseline_asr[attack_lang] = {
        ml: float((preds_attacked[attack_lang][ml] == target).mean()) for ml in LANGS
    }
    print(f'{time.time()-t0:.1f}s')
    print(f'  Attacked acc: { {k: f"{100*v:.1f}%" for k,v in baseline_acc[attack_lang].items()} }')
    print(f'  Attacked ASR: { {k: f"{100*v:.1f}%" for k,v in baseline_asr[attack_lang].items()} }')

print(f'Unilingual attack ready: {NUM_BOXES} boxes @ size {FONT_SIZE}')


Clean acc:    {'en': '85.9%', 'zh': '91.4%'}
Building en sole two-box attack... 5.6s
  Attacked acc: {'en': '3.4%', 'zh': '27.1%'}
  Attacked ASR: {'en': '96.5%', 'zh': '71.8%'}
Building zh sole two-box attack... 5.5s
  Attacked acc: {'en': '72.3%', 'zh': '39.6%'}
  Attacked ASR: {'en': '1.6%', 'zh': '58.8%'}
Unilingual attack ready: 2 boxes @ size 24


## 3. Saliency functions

Each variant exposes **two entry points**:

| Function | Purpose | Forward passes |
|---|---|---|
| `attn_map_en/zh(img, variant)` | heatmap only — used for visualization | 1 |
| `classify_and_attn_en/zh(img, variant)` | **(pred, cam) fused** — used in defense loop | 1 |
| `classify_and_gradcam_en/zh(img)` | **(pred, cam) fused** — GradCAM reference | 1 fwd + 1 back |

The GradCAM fused version is **still counted as 2 passes** in the cost table because the
backward pass is as expensive as a forward pass on modern hardware (same memory, same ops).

In [6]:
def _norm_cam(cam):
    cam = np.maximum(cam if isinstance(cam, np.ndarray) else cam.cpu().numpy(), 0)
    cam -= cam.min()
    mx   = cam.max()
    return cam / mx if mx > 0 else cam

def align_cam(cam, size=DISPLAY_SIZE):
    return np.array(
        Image.fromarray((cam*255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    ) / 255.0

# ─── shared attention helpers ─────────────────────────────────────────────────

def _make_en_hook(collector):
    """Hook factory for open_clip nn.MultiheadAttention blocks."""
    def hook(module, inputs, output):
        q_in = inputs[0]   # (L, B, D) or (B, L, D) depending on batch_first
        if getattr(module, 'batch_first', False):
            B, L, D = q_in.shape
        else:
            L, B, D = q_in.shape
            q_in = q_in.transpose(0, 1).contiguous()   # → (B, L, D)
        n_head = module.num_heads
        hd     = D // n_head
        with torch.no_grad():
            qkv = F.linear(q_in, module.in_proj_weight, module.in_proj_bias)  # (B, L, 3D)
            q, k, _ = qkv.chunk(3, dim=-1)
            q = q.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)   # (B, nh, L, hd)
            k = k.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)
            attn = (q @ k.transpose(-2, -1)) * (hd ** -0.5)        # (B, nh, L, L)
            attn = attn.softmax(dim=-1)
        collector.append(attn[0].detach().cpu())   # (nh, L, L)
    return hook

def _build_attn_cam(all_attns, variant):
    """Turn collected per-layer attention tensors into a spatial cam."""
    if variant == 'last':
        cls_row = all_attns[-1].mean(0)[0, 1:]          # (n_patches,)
    else:   # rollout
        L = all_attns[0].shape[-1]
        rollout = torch.eye(L)
        for a in all_attns:
            a_r = 0.5 * a.mean(0) + 0.5 * torch.eye(L)
            a_r = a_r / a_r.sum(-1, keepdim=True)
            rollout = a_r @ rollout
        cls_row = rollout[0, 1:]
    n = int(round(cls_row.shape[0] ** 0.5))
    return _norm_cam(cls_row.reshape(n, n).numpy())

# ─── EN CLIP attention ────────────────────────────────────────────────────────

def attn_map_en(pil_img, variant='last'):
    """Heatmap only. 1 forward pass."""
    wrapper   = models['en']
    x         = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
    collector = []
    handles   = [rb.attn.register_forward_hook(_make_en_hook(collector))
                 for rb in wrapper.m.visual.transformer.resblocks]
    with torch.no_grad():
        wrapper.m.visual(x)
    for h in handles: h.remove()
    return _build_attn_cam(collector, variant)

def classify_and_attn_en(pil_img, variant='last'):
    """
    Fused: classification + attention in ONE forward pass.
    Returns (pred_int, cam_array).
    """
    wrapper   = models['en']
    x         = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
    collector = []
    handles   = [rb.attn.register_forward_hook(_make_en_hook(collector))
                 for rb in wrapper.m.visual.transformer.resblocks]
    with torch.no_grad():
        feat = wrapper.m.visual(x)
        imf  = F.normalize(feat, dim=-1)
        pred = int((imf @ TEXT_EMB['en'].T).squeeze().argmax().item())
    for h in handles: h.remove()
    return pred, _build_attn_cam(collector, variant)

# ─── ZH CLIP attention ────────────────────────────────────────────────────────

def attn_map_zh(pil_img, variant='last'):
    """Heatmap only. 1 forward pass via output_attentions=True."""
    wrapper = models['zh']
    pv = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        vis_out = wrapper.m.vision_model(pixel_values=pv, output_attentions=True)
    attns = [a[0].cpu() for a in vis_out.attentions]   # list of (nh, L, L)
    return _build_attn_cam(attns, variant)

def classify_and_attn_zh(pil_img, variant='last'):
    """
    Fused: classification + attention in ONE forward pass.
    ChineseCLIP: vision_model(output_attentions=True) returns attentions AND
    pooler_output; visual_projection maps it to the final CLIP embedding.
    Returns (pred_int, cam_array).
    """
    wrapper = models['zh']
    pv = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        vis_out  = wrapper.m.vision_model(pixel_values=pv, output_attentions=True)
        # visual_projection is the linear layer that maps pooler_output → CLIP embedding
        proj_out = wrapper.m.visual_projection(vis_out.pooler_output)
        imf      = F.normalize(proj_out, dim=-1)
        pred     = int((imf @ TEXT_EMB['zh'].T).squeeze().argmax().item())
    attns = [a[0].cpu() for a in vis_out.attentions]
    return pred, _build_attn_cam(attns, variant)

# ─── GradCAM (for comparison) ─────────────────────────────────────────────────

def _cam_from_conv(act, grad):
    w   = grad.mean(dim=(2,3), keepdim=True)
    cam = (w * act).sum(dim=1).squeeze(0)
    return _norm_cam(cam.relu().detach().cpu().numpy())

def classify_and_gradcam_en(pil_img):
    """
    GradCAM for EN CLIP. Costs 1 forward + 1 backward pass.
    Counted as 2 in the inference cost table.
    Returns (pred_int, cam_array).
    """
    wrapper = models['en']; acts = {}
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle  = wrapper.m.visual.conv1.register_forward_hook(hook)
    x       = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
    feat    = wrapper.m.visual(x)
    imf     = F.normalize(feat, dim=-1)
    sims    = (imf @ TEXT_EMB['en'].T).squeeze()
    pred    = int(sims.detach().argmax().item())
    sims[pred].backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); wrapper.m.zero_grad()
    return pred, cam

def classify_and_gradcam_zh(pil_img):
    """
    GradCAM for ZH CLIP. Costs 1 forward + 1 backward pass.
    Returns (pred_int, cam_array).
    """
    wrapper = models['zh']; acts = {}
    patch   = wrapper.m.vision_model.embeddings.patch_embedding
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle  = patch.register_forward_hook(hook)
    pv      = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    out     = wrapper.m.get_image_features(pixel_values=pv)
    imf     = F.normalize(_clip_feat(out), dim=-1)
    sims    = (imf @ TEXT_EMB['zh'].T).squeeze()
    pred    = int(sims.detach().argmax().item())
    sims[pred].backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); wrapper.m.zero_grad()
    return pred, cam

print('All saliency functions ready.')
print('  EN: 12 blocks, 8 heads, 7×7=49 patches')
print('  ZH: 12 blocks, 12 heads, 14×14=196 patches')

All saliency functions ready.
  EN: 12 blocks, 8 heads, 7×7=49 patches
  ZH: 12 blocks, 12 heads, 14×14=196 patches


## 4. Masking helpers

In [7]:
def n_cam_intersection(*cams):
    return np.minimum.reduce([align_cam(c) for c in cams])

def dilate_mask(mask, iterations=3):
    m = mask.astype(bool)
    for _ in range(iterations):
        pad = np.pad(m, 1, mode='constant', constant_values=False)
        m = (pad[:-2,:-2]|pad[:-2,1:-1]|pad[:-2,2:]|
             pad[1:-1,:-2]|pad[1:-1,1:-1]|pad[1:-1,2:]|
             pad[2:,:-2]  |pad[2:,1:-1]  |pad[2:,2:])
    return m

def cam_to_mask(saliency, threshold=0.85, dilate=3):
    thr  = np.percentile(saliency, threshold * 100)
    mask = saliency >= thr
    if dilate > 0: mask = dilate_mask(mask, iterations=dilate)
    return mask

def apply_mask(pil_img, mask):
    arr = np.array(pil_img.convert('RGB'))
    m   = mask.astype(bool)
    if mask.shape != arr.shape[:2]:
        m = np.array(Image.fromarray(m.astype(np.uint8)*255).resize(
                     arr.shape[1::-1], Image.NEAREST)) > 127
    out  = arr.copy()
    mean = arr[~m].mean(0) if (~m).any() else arr.reshape(-1,3).mean(0)
    out[m] = mean
    return Image.fromarray(out.astype(np.uint8))

def overlay_cam(pil_img, cam, alpha=0.5):
    c224  = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize(
                     (DISPLAY_SIZE, DISPLAY_SIZE), Image.BILINEAR)) / 255.0
    heat  = cm.jet(c224)[:,:,:3]
    base  = np.array(pil_img.resize((DISPLAY_SIZE, DISPLAY_SIZE))).astype(np.float32)/255.0
    return Image.fromarray((np.clip((1-alpha)*base + alpha*heat, 0, 1)*255).astype(np.uint8))

print('Masking helpers ready.')

Masking helpers ready.


In [8]:
OCCL_GRID = {'en': 7, 'zh': 14}   # match ViT patch grids (B/32 → 7×7, B/16 → 14×14)

def _occ_fill(pil_img):
    arr = np.array(pil_img.convert('RGB'))
    return arr.reshape(-1, 3).mean(0).astype(np.uint8)

def _occlude_cell(pil_img, row, col, grid, fill):
    arr = np.array(pil_img.convert('RGB'))
    cell = DISPLAY_SIZE // grid
    arr[row*cell:(row+1)*cell, col*cell:(col+1)*cell] = fill
    return Image.fromarray(arr)

@torch.no_grad()
def occlusion_map_en(pil_img, grid=None):
    """Drop in predicted-class cosine when each grid cell is mean-filled. 1 + G² fwd."""
    grid = grid or OCCL_GRID['en']
    wrapper = models['en']
    imf = wrapper.embed_images([pil_img])
    pred = int((imf @ TEXT_EMB['en'].T).squeeze().argmax().item())
    base = (imf @ TEXT_EMB['en'][pred:pred+1].T).squeeze().item()
    fill = _occ_fill(pil_img)
    occ_imgs = [_occlude_cell(pil_img, *divmod(k, grid), grid, fill) for k in range(grid * grid)]
    scores = (wrapper.embed_images(occ_imgs) @ TEXT_EMB['en'][pred:pred+1].T).squeeze()
    return _norm_cam((base - scores).clamp(min=0).reshape(grid, grid).cpu().numpy())

@torch.no_grad()
def occlusion_map_zh(pil_img, grid=None):
    """Drop in predicted-class cosine when each grid cell is mean-filled. 1 + G² fwd."""
    grid = grid or OCCL_GRID['zh']
    wrapper = models['zh']
    imf = wrapper.embed_images([pil_img])
    pred = int((imf @ TEXT_EMB['zh'].T).squeeze().argmax().item())
    base = (imf @ TEXT_EMB['zh'][pred:pred+1].T).squeeze().item()
    fill = _occ_fill(pil_img)
    occ_imgs = [_occlude_cell(pil_img, *divmod(k, grid), grid, fill) for k in range(grid * grid)]
    scores = (wrapper.embed_images(occ_imgs) @ TEXT_EMB['zh'][pred:pred+1].T).squeeze()
    return _norm_cam((base - scores).clamp(min=0).reshape(grid, grid).cpu().numpy())

print('Occlusion saliency ready.')
print(f'  EN: {OCCL_GRID["en"]}×{OCCL_GRID["en"]} grid')
print(f'  ZH: {OCCL_GRID["zh"]}×{OCCL_GRID["zh"]} grid')

Occlusion saliency ready.
  EN: 7×7 grid
  ZH: 14×14 grid


## 5. Heatmap comparison — 5 example images

In [9]:
example_indices = [int(np.where(true == c)[0][0]) for c in range(5)]
# Viz on EN sole two-box attack (representative)
_viz_imgs = attacked_by_lang['en']

col_titles = ['Attacked',
              'GradCAM EN', 'GradCAM ZH',
              'Attn-last EN', 'Attn-last ZH',
              'Rollout EN', 'Rollout ZH']

fig, axes = plt.subplots(len(example_indices), len(col_titles),
                          figsize=(len(col_titles)*2.4, len(example_indices)*2.6))
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=8, fontweight='bold')

for row_i, pos in enumerate(example_indices):
    img = _viz_imgs[pos]
    _, gcam_en = classify_and_gradcam_en(img)
    _, gcam_zh = classify_and_gradcam_zh(img)
    panels = [
        img,
        overlay_cam(img, gcam_en),
        overlay_cam(img, gcam_zh),
        overlay_cam(img, attn_map_en(img, 'last')),
        overlay_cam(img, attn_map_zh(img, 'last')),
        overlay_cam(img, attn_map_en(img, 'rollout')),
        overlay_cam(img, attn_map_zh(img, 'rollout')),
    ]
    for col_i, panel in enumerate(panels):
        axes[row_i, col_i].imshow(panel); axes[row_i, col_i].axis('off')
    axes[row_i, 0].set_ylabel(CLASSES['en'][true[pos]], fontsize=8,
                               rotation=0, labelpad=42, va='center')

plt.suptitle('GradCAM vs attention maps — unilingual EN dual-box attack', fontsize=11, y=1.002)
plt.tight_layout()
plt.savefig('results/heatmap_comparison.png', dpi=130, bbox_inches='tight')
plt.close()
print('Saved -> results/heatmap_comparison.png')


Saved -> results/heatmap_comparison.png


## 5b. Occlusion map comparison — same 5 examples

In [10]:
example_indices = [int(np.where(true == c)[0][0]) for c in range(5)]
_viz_imgs = attacked_by_lang['en']
occ_col_titles = ['Attacked', 'Occlusion EN', 'Occlusion ZH']

fig, axes = plt.subplots(len(example_indices), len(occ_col_titles),
                          figsize=(len(occ_col_titles)*2.4, len(example_indices)*2.6))
for ax, title in zip(axes[0], occ_col_titles):
    ax.set_title(title, fontsize=8, fontweight='bold')

for row_i, pos in enumerate(example_indices):
    img = _viz_imgs[pos]
    panels = [
        img,
        overlay_cam(img, occlusion_map_en(img)),
        overlay_cam(img, occlusion_map_zh(img)),
    ]
    for col_i, panel in enumerate(panels):
        axes[row_i, col_i].imshow(panel); axes[row_i, col_i].axis('off')
    axes[row_i, 0].set_ylabel(CLASSES['en'][true[pos]], fontsize=8,
                               rotation=0, labelpad=42, va='center')

plt.suptitle('Occlusion sensitivity maps — unilingual EN dual-box attack', fontsize=11, y=1.002)
plt.tight_layout()
plt.savefig('results/occlusion_comparison.png', dpi=130, bbox_inches='tight')
plt.close()
print('Saved -> results/occlusion_comparison.png')


Saved -> results/occlusion_comparison.png


## 6. Defense evaluation

All variants use the **fused** `classify_and_*` functions so the inference cost
is correctly counted.

### Inference cost per image

```
GradCAM cam_2mod : 2 (fused fwd+back EN/ZH) + 2 (re-classify masked) = 6
Attn    cam_2mod : 2 (fused fwd EN/ZH)       + 2 (re-classify masked) = 4
```

In [11]:
THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]

# Each variant is (name, fused_fn_en, fused_fn_zh, inference_cost)
# fused_fn_*: callable(pil_img) -> (pred_int, cam_array)
VARIANTS = [
    ('GradCAM',
     classify_and_gradcam_en,
     classify_and_gradcam_zh,
     6),   # 2×(fwd+back) + 2×fwd
    ('Attn-last',
     lambda img: classify_and_attn_en(img, 'last'),
     lambda img: classify_and_attn_zh(img, 'last'),
     4),   # 2×fwd + 2×fwd
    ('Attn-rollout',
     lambda img: classify_and_attn_en(img, 'rollout'),
     lambda img: classify_and_attn_zh(img, 'rollout'),
     4),   # 2×fwd + 2×fwd
]

print('Variants and inference costs:')
for name, _, _, cost in VARIANTS:
    print(f'  {name:<16} {cost} forward passes/image')

Variants and inference costs:
  GradCAM          6 forward passes/image
  Attn-last        4 forward passes/image
  Attn-rollout     4 forward passes/image


In [12]:
def run_defense(name, fn_en, fn_zh, cost, indices, images, label=''):
    """
    Run the 2-mod intersection defense on `images[indices]`.
    Uses fused fn_en/fn_zh that each return (pred, cam) in ONE forward pass.

    Defense loop per image:
      1. fn_en(img) → (pred_en, cam_en)   [1 pass]
      2. fn_zh(img) → (pred_zh, cam_zh)   [1 pass]
      3. compute intersection, apply mask
      4. classify masked with EN model     [1 pass]
      5. classify masked with ZH model     [1 pass]
    Total: `cost` passes (4 for attn, 6 for gradcam).
    """
    imgs_sub  = [images[i] for i in indices]
    true_sub  = true[indices]
    tgt_sub   = target[indices]
    n         = len(indices)

    print(f'  [{name}] {label} n={n}...', end=' ', flush=True)
    t0 = time.time()

    preds_en, preds_zh = [], []
    saliency_maps = []
    for img in imgs_sub:
        pred_en, cam_en = fn_en(img)   # pass 1
        pred_zh, cam_zh = fn_zh(img)   # pass 2
        preds_en.append(pred_en)
        preds_zh.append(pred_zh)
        saliency_maps.append(n_cam_intersection(cam_en, cam_zh))

    preds_en = np.array(preds_en)
    preds_zh = np.array(preds_zh)
    heatmap_time = time.time() - t0

    return preds_en, preds_zh, saliency_maps, true_sub, tgt_sub, heatmap_time


def apply_and_classify(saliency_maps, images_sub, indices_sub, threshold):
    """Apply mask at `threshold` and re-classify with both models (passes 3+4)."""
    masks       = [cam_to_mask(s, threshold, dilate=3) for s in saliency_maps]
    masked_imgs = [apply_mask(images_sub[j], masks[j]) for j in range(len(images_sub))]
    coverage    = float(np.mean([m.mean() for m in masks]))
    new_preds   = {ml: classify_batch(models[ml], masked_imgs, CLASSES[ml]) for ml in LANGS}
    return new_preds, coverage


print('Defense helpers defined.')

Defense helpers defined.


## 7. Threshold sweep — tune subset (100 images), per attack lang

Tune GradCAM / Attn-last / Attn-rollout thresholds separately for EN-only and ZH-only
dual-box attacks. Pick best thr by EN masked accuracy (same rule as the unilingual notebook).


In [13]:
sweep_results   = {}   # attack_lang → variant → list of sweep rows
best_thresholds = {}   # attack_lang → variant → thr
precomputed     = {}   # attack_lang → variant → saliency tuple

for attack_lang in LANGS:
    print(f'\n=== Tune on {attack_lang} sole two-box (n={len(tune_idx)}) ===')
    attacked_imgs = attacked_by_lang[attack_lang]
    tune_imgs = [attacked_imgs[i] for i in tune_idx]
    sweep_results[attack_lang] = {}
    best_thresholds[attack_lang] = {}
    precomputed[attack_lang] = {}

    for name, fn_en, fn_zh, cost in VARIANTS:
        print(f'\nVariant: {name}  (cost={cost})')
        preds_en, preds_zh, salmaps, true_sub, tgt_sub, t_heat = run_defense(
            name, fn_en, fn_zh, cost, tune_idx, attacked_imgs, 'tune')
        precomputed[attack_lang][name] = (preds_en, preds_zh, salmaps, true_sub, tgt_sub)
        print(f'{t_heat:.1f}s ({1000*t_heat/len(tune_idx):.0f}ms/img)')

        rows = []
        for thr in THRESHOLDS:
            new_preds, cov = apply_and_classify(salmaps, tune_imgs, tune_idx, thr)
            for ml in LANGS:
                rows.append({
                    'variant': name, 'model': ml, 'threshold': thr,
                    'masked_acc': float((new_preds[ml] == true_sub).mean()),
                    'masked_asr': float((new_preds[ml] == tgt_sub).mean()),
                    'coverage': cov,
                })
        sweep_results[attack_lang][name] = rows
        best = max([r for r in rows if r['model']=='en'], key=lambda r: r['masked_acc'])
        best_thresholds[attack_lang][name] = best['threshold']
        print(f'  best thr={best["threshold"]}  '
              f'EN {100*best["masked_acc"]:.1f}%  coverage {100*best["coverage"]:.1f}%')

print('\nBest thresholds:')
for al in LANGS:
    for name, thr in best_thresholds[al].items():
        print(f'  {al}/{name}: {thr}')



=== Tune on en sole two-box (n=100) ===

Variant: GradCAM  (cost=6)
  [GradCAM] tune n=100... 6.2s (62ms/img)
  best thr=0.8  EN 24.0%  coverage 36.2%

Variant: Attn-last  (cost=4)
  [Attn-last] tune n=100... 2.9s (29ms/img)
  best thr=0.95  EN 66.0%  coverage 8.5%

Variant: Attn-rollout  (cost=4)
  [Attn-rollout] tune n=100... 3.3s (33ms/img)
  best thr=0.85  EN 45.0%  coverage 21.7%

=== Tune on zh sole two-box (n=100) ===

Variant: GradCAM  (cost=6)
  [GradCAM] tune n=100... 6.1s (61ms/img)
  best thr=0.95  EN 53.0%  coverage 9.5%

Variant: Attn-last  (cost=4)
  [Attn-last] tune n=100... 3.0s (30ms/img)
  best thr=0.85  EN 59.0%  coverage 23.2%

Variant: Attn-rollout  (cost=4)
  [Attn-rollout] tune n=100... 3.4s (34ms/img)
  best thr=0.9  EN 56.0%  coverage 15.0%

Best thresholds:
  en/GradCAM: 0.8
  en/Attn-last: 0.95
  en/Attn-rollout: 0.85
  zh/GradCAM: 0.95
  zh/Attn-last: 0.85
  zh/Attn-rollout: 0.9


In [14]:
# Plot threshold sweeps — one row per attack lang
fig, axes = plt.subplots(len(LANGS), len(VARIANTS),
                         figsize=(5*len(VARIANTS), 4*len(LANGS)), sharey=True)
if len(LANGS) == 1:
    axes = np.array([axes])
for row, attack_lang in enumerate(LANGS):
    for col, (name, _, _, cost) in enumerate(VARIANTS):
        ax = axes[row, col]
        for ml, color in zip(LANGS, ['C0', 'C1']):
            sub = [r for r in sweep_results[attack_lang][name] if r['model']==ml]
            xs  = [r['threshold'] for r in sub]
            ax.plot(xs, [100*r['masked_acc'] for r in sub], 'o-', color=color, label=f'{ml} acc')
            ax.plot(xs, [100*r['masked_asr'] for r in sub], 's--', color=color, alpha=0.5, label=f'{ml} ASR')
        bt = best_thresholds[attack_lang][name]
        ax.axvline(bt, color='gray', lw=1, ls=':', label=f'best={bt}')
        ax.set_title(f'{attack_lang} / {name} (cost={cost})', fontsize=9)
        ax.set_xlabel('Percentile threshold')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    axes[row, 0].set_ylabel(f'{attack_lang} attack — % (tune n=100)')
plt.suptitle('Threshold sweep — unilingual sole two-box', fontsize=11)
plt.tight_layout()
plt.savefig('results/threshold_sweep_comparison.png', dpi=140, bbox_inches='tight')
plt.close()
print('Saved -> results/threshold_sweep_comparison.png')


Saved -> results/threshold_sweep_comparison.png


## 8. Full eval on 1000 images — run if promising

Use the best thresholds from the 100-image tune above.
**Skip this section** if tune results are not promising; otherwise run per attack lang
and write `results/confusion_results_{variant}_{attack_lang}.json`.


In [15]:
full_results = {}  # attack_lang → variant → results dict

for attack_lang in LANGS:
    print(f'\n======== Full eval: {attack_lang} sole two-box (n={len(all_idx)}) ========')
    attacked_imgs = attacked_by_lang[attack_lang]
    full_results[attack_lang] = {}

    for name, fn_en, fn_zh, cost in VARIANTS:
        best_thr = best_thresholds[attack_lang][name]
        print(f'\n[{name}] full eval (n={len(all_idx)}, cost={cost}, thr={best_thr})...')

        preds_en, preds_zh, salmaps, _, _, t_heat = run_defense(
            name, fn_en, fn_zh, cost, all_idx, attacked_imgs, 'full')
        print(f'  heatmaps: {t_heat:.0f}s ({1000*t_heat/len(all_idx):.0f}ms/img)')

        new_preds, coverage = apply_and_classify(
            salmaps, attacked_imgs, all_idx, best_thr)

        # Clean-image degradation
        _, _, clean_sals, _, _, _ = run_defense(
            name, fn_en, fn_zh, cost, all_idx, clean_224, 'clean')
        clean_new, _ = apply_and_classify(clean_sals, clean_224, all_idx, best_thr)

        defense_res = {}
        for ml, preds_atk in [('en', preds_en), ('zh', preds_zh)]:
            wrong = np.array(preds_atk) != true
            defense_res[ml] = {
                'acc':           float((new_preds[ml] == true).mean()),
                'asr':           float((new_preds[ml] == target).mean()),
                'recovery_rate': float((wrong & (new_preds[ml]==true)).sum()/wrong.sum()) if wrong.any() else 0.0,
                'baseline_acc':  baseline_acc[attack_lang][ml],
                'baseline_asr':  baseline_asr[attack_lang][ml],
            }

        clean_deg = {
            ml: {
                'baseline_acc': clean_acc[ml],
                'masked_acc':   float((clean_new[ml] == true).mean()),
                'delta_acc':    float((clean_new[ml] == true).mean()) - clean_acc[ml],
            } for ml in LANGS
        }

        results = {
            'setup':             'unilingual',
            'method':            f'attn_2mod_{name.lower().replace("-","_")}',
            'variant':           name,
            'attack':            attack_lang,
            'n_images':          int(len(all_idx)),
            'best_threshold':    best_thr,
            'inference_cost':    cost,
            'heatmap_time_s':    round(t_heat, 2),
            'time_per_image_ms': round(1000*t_heat/len(all_idx), 1),
            'clean_acc':         clean_acc,
            'baseline_acc':      {ml: defense_res[ml]['baseline_acc'] for ml in LANGS},
            'baseline_asr':      {ml: defense_res[ml]['baseline_asr'] for ml in LANGS},
            'defense':           defense_res,
            'clean_degradation': clean_deg,
            'coverage_mean':     coverage,
            'defense_acc_mean':  float(np.mean([defense_res[ml]['acc']  for ml in LANGS])),
            'defense_asr_mean':  float(np.mean([defense_res[ml]['asr']  for ml in LANGS])),
        }
        slug = name.lower().replace('-', '_').replace(' ', '_')
        out  = f'results/confusion_results_{slug}_{attack_lang}.json'
        with open(out, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        full_results[attack_lang][name] = results
        for ml in LANGS:
            d = defense_res[ml]
            print(f'  model_{ml}: {100*d["baseline_acc"]:.1f}% → {100*d["acc"]:.1f}%  '
                  f'ASR {100*d["baseline_asr"]:.1f}% → {100*d["asr"]:.1f}%  '
                  f'recovery={100*d["recovery_rate"]:.1f}%')
        print(f'  mean acc={100*results["defense_acc_mean"]:.1f}%  '
              f'coverage={100*coverage:.1f}%  '
              f'Saved -> {out}')



======== Full eval: en sole two-box (n=1000) ========

[GradCAM] full eval (n=1000, cost=6, thr=0.8)...
  [GradCAM] full n=1000...   heatmaps: 64s (64ms/img)
  [GradCAM] clean n=1000...   model_en: 3.4% → 22.1%  ASR 96.5% → 41.7%  recovery=21.4%
  model_zh: 27.1% → 35.4%  ASR 71.8% → 31.7%  recovery=28.1%
  mean acc=28.7%  coverage=35.8%  Saved -> results/confusion_results_gradcam_en.json

[Attn-last] full eval (n=1000, cost=4, thr=0.95)...
  [Attn-last] full n=1000...   heatmaps: 19s (19ms/img)
  [Attn-last] clean n=1000...   model_en: 3.4% → 63.2%  ASR 96.5% → 6.5%  recovery=62.2%
  model_zh: 27.1% → 72.0%  ASR 71.8% → 3.8%  recovery=64.3%
  mean acc=67.6%  coverage=8.5%  Saved -> results/confusion_results_attn_last_en.json

[Attn-rollout] full eval (n=1000, cost=4, thr=0.85)...
  [Attn-rollout] full n=1000...   heatmaps: 19s (19ms/img)
  [Attn-rollout] clean n=1000...   model_en: 3.4% → 48.2%  ASR 96.5% → 13.9%  recovery=46.8%
  model_zh: 27.1% → 64.1%  ASR 71.8% → 10.6%  recovery=

## 9. Summary

In [16]:
# Placement dual-box GradCAM cam_2mod (intersection_min) reference
# from en_zh_multiple_placement/results/cam_defense/confusion_results_cam_defense.json
placement_ref = {
    'en': {'en_acc': 0.222, 'zh_acc': 0.354, 'mean_acc': 0.288,
           'coverage': 0.358, 'cost': 6, 'thr': 0.80},
    'zh': {'en_acc': 0.388, 'zh_acc': 0.353, 'mean_acc': 0.3705,
           'coverage': 0.351, 'cost': 6, 'thr': 0.80},
}
# Multilingual Attn-last from parent notebook (context only)
multi_attn_last = {'en_acc': 0.687, 'zh_acc': 0.765, 'mean_acc': 0.726,
                   'coverage': 0.077, 'cost': 4, 'thr': 0.95}

for attack_lang in LANGS:
    ref = placement_ref[attack_lang]
    print('\n' + '='*76)
    print(f'  {attack_lang} sole two-box — Attention vs GradCAM')
    print(f'  {"Method":<22}  {"Cost":>5}  {"EN acc":>7}  {"ZH acc":>7}  {"Mean":>7}  {"Cov":>6}  {"Thr":>5}')
    print('='*76)
    print(f'  {"placement GradCAM":<22}  '
          f'{ref["cost"]:>5}  '
          f'{100*ref["en_acc"]:>6.1f}%  '
          f'{100*ref["zh_acc"]:>6.1f}%  '
          f'{100*ref["mean_acc"]:>6.1f}%  '
          f'{100*ref["coverage"]:>5.1f}%  '
          f'{ref["thr"]:>5.2f}')
    for name, res in full_results[attack_lang].items():
        print(f'  {name:<22}  '
              f'{res["inference_cost"]:>5}  '
              f'{100*res["defense"]["en"]["acc"]:>6.1f}%  '
              f'{100*res["defense"]["zh"]["acc"]:>6.1f}%  '
              f'{100*res["defense_acc_mean"]:>6.1f}%  '
              f'{100*res["coverage_mean"]:>5.1f}%  '
              f'{best_thresholds[attack_lang][name]:>5.2f}')
    print('='*76)

    print('Clean degradation:')
    print(f'  {"Method":<22}  {"EN baseline→masked":>22}  {"ZH baseline→masked":>22}')
    for name, res in full_results[attack_lang].items():
        cd = res['clean_degradation']
        print(f'  {name:<22}  '
              f'{100*cd["en"]["baseline_acc"]:>5.1f}% → {100*cd["en"]["masked_acc"]:>5.1f}% ({100*cd["en"]["delta_acc"]:>+.1f}pp)  '
              f'{100*cd["zh"]["baseline_acc"]:>5.1f}% → {100*cd["zh"]["masked_acc"]:>5.1f}% ({100*cd["zh"]["delta_acc"]:>+.1f}pp)')

print(f'\n(Context) multilingual Attn-last mean={100*multi_attn_last["mean_acc"]:.1f}% '
      f'cost={multi_attn_last["cost"]}')



  en sole two-box — Attention vs GradCAM
  Method                   Cost   EN acc   ZH acc     Mean     Cov    Thr
  placement GradCAM           6    22.2%    35.4%    28.8%   35.8%   0.80
  GradCAM                     6    22.1%    35.4%    28.7%   35.8%   0.80
  Attn-last                   4    63.2%    72.0%    67.6%    8.5%   0.95
  Attn-rollout                4    48.2%    64.1%    56.1%   21.6%   0.85
Clean degradation:
  Method                      EN baseline→masked      ZH baseline→masked
  GradCAM                  85.9% →  41.7% (-44.2pp)   91.4% →  55.2% (-36.2pp)
  Attn-last                85.9% →  77.1% (-8.8pp)   91.4% →  88.8% (-2.6pp)
  Attn-rollout             85.9% →  60.3% (-25.6pp)   91.4% →  74.6% (-16.8pp)

  zh sole two-box — Attention vs GradCAM
  Method                   Cost   EN acc   ZH acc     Mean     Cov    Thr
  placement GradCAM           6    38.8%    35.3%    37.0%   35.1%   0.80
  GradCAM                     6    58.4%    43.5%    50.9%    9.4%   0.

In [17]:
# Summary bar charts — one panel per attack lang
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, attack_lang in zip(axes, LANGS):
    ref = placement_ref[attack_lang]
    names = ['GradCAM\n(placement)'] + [
        f'{n}\n(cost={full_results[attack_lang][n]["inference_cost"]})'
        for n, _, _, _ in VARIANTS
    ]
    en_accs = [ref['en_acc']] + [full_results[attack_lang][n]['defense']['en']['acc']
                                 for n, _, _, _ in VARIANTS]
    zh_accs = [ref['zh_acc']] + [full_results[attack_lang][n]['defense']['zh']['acc']
                                 for n, _, _, _ in VARIANTS]
    mn_accs = [ref['mean_acc']] + [full_results[attack_lang][n]['defense_acc_mean']
                                   for n, _, _, _ in VARIANTS]

    x = np.arange(len(names)); w = 0.27
    ax.bar(x-w, [100*v for v in en_accs], w, label='EN acc')
    ax.bar(x,   [100*v for v in zh_accs], w, label='ZH acc')
    ax.bar(x+w, [100*v for v in mn_accs], w, label='Mean acc', color='C2')
    if attack_lang == 'en':
        ax.axhline(100*multi_attn_last['mean_acc'], color='orange', ls=':',
                   label=f'multi Attn-last ({100*multi_attn_last["mean_acc"]:.0f}%)')
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=8)
    ax.set_ylabel('Post-defense accuracy (%)')
    ax.set_title(f'{attack_lang} sole two-box')
    ax.legend(fontsize=8); ax.grid(True, axis='y', alpha=0.3)

fig.suptitle('Attention vs GradCAM — unilingual dual-box (1000 images)', fontsize=11)
plt.tight_layout()
plt.savefig('results/final_comparison.png', dpi=140, bbox_inches='tight')
plt.close()
print('Saved -> results/final_comparison.png')


Saved -> results/final_comparison.png
